# Energy Forecasting Model

This notebook creates a machine learning model to forecast energy consumption based on CPU and memory metrics.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette('husl')

## 1. Data Loading and Exploration

In [ ]:
# Load the data
df = pd.read_csv('node_metrics_export.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics
print("Dataset Info:")
print(df.info())
print("\nBasic Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

## 2. Data Preprocessing and Feature Engineering

In [ ]:
# Convert timestamp to datetime (for sorting and visualization only)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Drop columns we don't want to use in the model
columns_to_drop = ['created_at', 'memory_utilization_bytes', 'memory_assigned_bytes', 'machine_memory_total_bytes']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Create derived features
df['cpu_memory_ratio'] = df['cpu_utilization_percent'] / (df['memory_utilization_percent'] + 1e-8)
df['total_power_consumption'] = df['cpu_package_watts'] + df['memory_power_watts'] + df['platform_watts']

# Sort by timestamp
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Processed dataset shape: {df.shape}")
print(f"Columns after preprocessing: {list(df.columns)}")
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Plot energy consumption over time
plt.figure(figsize=(15, 8))

plt.subplot(2, 2, 1)
plt.plot(df['timestamp'], df['energy_watts'])
plt.title('Energy Consumption Over Time')
plt.xlabel('Time')
plt.ylabel('Energy (Watts)')
plt.xticks(rotation=45)

plt.subplot(2, 2, 2)
plt.plot(df['timestamp'], df['cpu_utilization_percent'])
plt.title('CPU Utilization Over Time')
plt.xlabel('Time')
plt.ylabel('CPU Utilization (%)')
plt.xticks(rotation=45)

plt.subplot(2, 2, 3)
plt.plot(df['timestamp'], df['memory_utilization_percent'])
plt.title('Memory Utilization Over Time')
plt.xlabel('Time')
plt.ylabel('Memory Utilization (%)')
plt.xticks(rotation=45)

plt.subplot(2, 2, 4)
plt.plot(df['timestamp'], df['total_power_consumption'])
plt.title('Total Power Consumption Over Time')
plt.xlabel('Time')
plt.ylabel('Power (Watts)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
plt.figure(figsize=(12, 10))
correlation_features = ['cpu_utilization_percent', 'memory_utilization_percent', 
                       'cpu_package_watts', 'memory_power_watts', 'platform_watts', 
                       'energy_watts', 'total_power_consumption', 'cpu_memory_ratio']

correlation_matrix = df[correlation_features].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots to understand relationships with energy consumption
plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.scatter(df['cpu_utilization_percent'], df['energy_watts'], alpha=0.6)
plt.xlabel('CPU Utilization (%)')
plt.ylabel('Energy (Watts)')
plt.title('CPU vs Energy')

plt.subplot(2, 3, 2)
plt.scatter(df['memory_utilization_percent'], df['energy_watts'], alpha=0.6)
plt.xlabel('Memory Utilization (%)')
plt.ylabel('Energy (Watts)')
plt.title('Memory vs Energy')

plt.subplot(2, 3, 3)
plt.scatter(df['cpu_package_watts'], df['energy_watts'], alpha=0.6)
plt.xlabel('CPU Package Watts')
plt.ylabel('Energy (Watts)')
plt.title('CPU Power vs Energy')

plt.subplot(2, 3, 4)
plt.scatter(df['memory_power_watts'], df['energy_watts'], alpha=0.6)
plt.xlabel('Memory Power Watts')
plt.ylabel('Energy (Watts)')
plt.title('Memory Power vs Energy')

plt.subplot(2, 3, 5)
plt.scatter(df['total_power_consumption'], df['energy_watts'], alpha=0.6)
plt.xlabel('Total Power Consumption')
plt.ylabel('Energy (Watts)')
plt.title('Total Power vs Energy')

plt.subplot(2, 3, 6)
plt.scatter(df['cpu_memory_ratio'], df['energy_watts'], alpha=0.6)
plt.xlabel('CPU/Memory Ratio')
plt.ylabel('Energy (Watts)')
plt.title('CPU/Memory Ratio vs Energy')

plt.tight_layout()
plt.show()

## 4. Feature Selection and Model Preparation

In [ ]:
# Select features for modeling
# Using utilization-based approach - most practical for forecasting
feature_columns = [
    'cpu_utilization_percent',
    'memory_utilization_percent',
    'cpu_memory_ratio'
]

# Alternative feature sets (uncomment to try):
# 
# Option 2: Power-component based
# feature_columns = [
#     'cpu_package_watts',
#     'memory_power_watts', 
#     'platform_watts'
# ]
#
# Option 3: Hybrid approach
# feature_columns = [
#     'cpu_utilization_percent',
#     'memory_utilization_percent',
#     'cpu_package_watts',
#     'memory_power_watts',
#     'platform_watts',
#     'cpu_memory_ratio'
# ]

# Prepare feature matrix X and target variable y
X = df[feature_columns].copy()
y = df['energy_watts'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used: {feature_columns}")
print(f"\nThis model predicts energy consumption based on system utilization metrics.")

In [ ]:
# Create lag features for time series forecasting
def create_lag_features(df, target_col, lags=[1, 2, 3, 5, 10]):
    """
    Create lag features for time series forecasting
    """
    df_lag = df.copy()
    
    for lag in lags:
        df_lag[f'{target_col}_lag_{lag}'] = df_lag[target_col].shift(lag)
    
    # Add rolling statistics
    for window in [5, 10, 20]:
        df_lag[f'{target_col}_rolling_mean_{window}'] = df_lag[target_col].rolling(window=window).mean()
        df_lag[f'{target_col}_rolling_std_{window}'] = df_lag[target_col].rolling(window=window).std()
    
    return df_lag

# Create enhanced dataset with lag features
df_enhanced = create_lag_features(df, 'energy_watts')

# Update feature columns to include lag features
lag_features = [col for col in df_enhanced.columns if 'energy_watts_lag' in col or 'energy_watts_rolling' in col]
enhanced_features = feature_columns + lag_features

# Remove rows with NaN values (due to lag features)
df_enhanced = df_enhanced.dropna()

X_enhanced = df_enhanced[enhanced_features].copy()
y_enhanced = df_enhanced['energy_watts'].copy()

print(f"Enhanced dataset shape after removing NaN: {df_enhanced.shape}")
print(f"Enhanced feature matrix shape: {X_enhanced.shape}")
print(f"Number of lag features added: {len(lag_features)}")

## 5. Model Training and Evaluation

In [ ]:
# Split data into training and testing sets (time-based split)
# Use last 20% of data for testing to maintain temporal order
split_idx = int(0.8 * len(X_enhanced))

X_train = X_enhanced.iloc[:split_idx]
X_test = X_enhanced.iloc[split_idx:]
y_train = y_enhanced.iloc[:split_idx]
y_test = y_enhanced.iloc[split_idx:]

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Define models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate models
model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled features for Linear Regression, original for tree-based models
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    model_results[name] = {
        'model': model,
        'predictions': y_pred,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }
    
    print(f"{name} Results:")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  R²: {r2:.4f}")

In [ ]:
# Create results comparison DataFrame
results_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'RMSE': [model_results[model]['rmse'] for model in model_results],
    'MAE': [model_results[model]['mae'] for model in model_results],
    'R²': [model_results[model]['r2'] for model in model_results]
})

print("\nModel Comparison:")
print(results_df.round(4))

# Find best model
best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
print(f"\nBest performing model: {best_model_name}")

## 6. Model Visualization and Analysis

In [ ]:
# Plot predictions vs actual values for all models
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, (name, results) in enumerate(model_results.items()):
    if i < 3:  # Only plot first 3 models
        axes[i].scatter(y_test, results['predictions'], alpha=0.6)
        axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[i].set_xlabel('Actual Energy (Watts)')
        axes[i].set_ylabel('Predicted Energy (Watts)')
        axes[i].set_title(f'{name}\nR² = {results["r2"]:.4f}')

# Model comparison bar plot
axes[3].bar(results_df['Model'], results_df['R²'])
axes[3].set_xlabel('Model')
axes[3].set_ylabel('R² Score')
axes[3].set_title('Model Performance Comparison')
axes[3].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Time series plot of predictions vs actual
plt.figure(figsize=(15, 8))

# Get test timestamps
test_timestamps = df_enhanced.iloc[split_idx:]['timestamp'].values

plt.plot(test_timestamps, y_test.values, label='Actual', linewidth=2, alpha=0.8)

for name, results in model_results.items():
    plt.plot(test_timestamps, results['predictions'], 
             label=f'{name} (R²={results["r2"]:.3f})', 
             linewidth=1.5, alpha=0.7)

plt.xlabel('Time')
plt.ylabel('Energy (Watts)')
plt.title('Energy Forecasting: Actual vs Predicted Values Over Time')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for the best model (if it's tree-based)
best_model = model_results[best_model_name]['model']

if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': enhanced_features,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
    plt.title(f'Top 15 Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))

## 7. Model Optimization

In [ ]:
# Hyperparameter tuning for the best model
if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
    base_model = RandomForestRegressor(random_state=42)
    
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.15],
        'max_depth': [3, 5, 7]
    }
    base_model = GradientBoostingRegressor(random_state=42)

else:
    print(f"Skipping hyperparameter tuning for {best_model_name}")
    base_model = None
    param_grid = None

if base_model is not None:
    print(f"Performing hyperparameter tuning for {best_model_name}...")
    
    grid_search = GridSearchCV(
        base_model, param_grid, cv=3, 
        scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {-grid_search.best_score_:.4f}")
    
    # Evaluate optimized model
    optimized_model = grid_search.best_estimator_
    y_pred_optimized = optimized_model.predict(X_test)
    
    rmse_optimized = np.sqrt(mean_squared_error(y_test, y_pred_optimized))
    r2_optimized = r2_score(y_test, y_pred_optimized)
    
    print(f"\nOptimized model performance:")
    print(f"  RMSE: {rmse_optimized:.4f}")
    print(f"  R²: {r2_optimized:.4f}")
    
    # Compare with original
    original_rmse = model_results[best_model_name]['rmse']
    original_r2 = model_results[best_model_name]['r2']
    
    print(f"\nImprovement:")
    print(f"  RMSE: {original_rmse:.4f} → {rmse_optimized:.4f} ({((original_rmse - rmse_optimized)/original_rmse)*100:.2f}% improvement)")
    print(f"  R²: {original_r2:.4f} → {r2_optimized:.4f} ({((r2_optimized - original_r2)/original_r2)*100:.2f}% improvement)")

## 8. Prediction Function for New Data

In [ ]:
def predict_energy_consumption(cpu_util, memory_util):
    """
    Predict energy consumption based on system utilization metrics
    
    Parameters:
    - cpu_util: CPU utilization percentage
    - memory_util: Memory utilization percentage  
    
    Returns:
    - predicted_energy: Predicted energy consumption in watts
    """
    # Calculate derived features
    cpu_memory_ratio = cpu_util / (memory_util + 1e-8)
    
    # Create input features array
    input_features = np.array([[
        cpu_util, memory_util, cpu_memory_ratio
    ]])
    
    # Use the best model for prediction
    basic_model = RandomForestRegressor(n_estimators=100, random_state=42)
    basic_model.fit(X_train[feature_columns], y_train)
    
    prediction = basic_model.predict(input_features)
    return prediction[0]

# Example usage
example_prediction = predict_energy_consumption(
    cpu_util=30.0,
    memory_util=15.0
)

print(f"\nExample prediction:")
print(f"CPU: 30%, Memory: 15%")
print(f"Expected energy consumption: {example_prediction:.2f} watts")

## 9. Model Saving

In [ ]:
# Save the best model
import joblib

# Save the optimized model if available, otherwise save the best original model
if 'optimized_model' in locals():
    model_to_save = optimized_model
    model_name = f"optimized_{best_model_name.lower().replace(' ', '_')}"
else:
    model_to_save = model_results[best_model_name]['model']
    model_name = f"{best_model_name.lower().replace(' ', '_')}"

# Save model and scaler
joblib.dump(model_to_save, f'energy_forecasting_{model_name}.pkl')
joblib.dump(scaler, 'energy_forecasting_scaler.pkl')

# Save feature names
with open('energy_forecasting_features.txt', 'w') as f:
    for feature in enhanced_features:
        f.write(f"{feature}\n")

print(f"Model saved as: energy_forecasting_{model_name}.pkl")
print(f"Scaler saved as: energy_forecasting_scaler.pkl")
print(f"Features saved as: energy_forecasting_features.txt")

## Summary

This notebook created a comprehensive energy forecasting model with the following key results:

1. **Data Analysis**: Analyzed relationship between CPU/memory metrics and energy consumption
2. **Feature Engineering**: Created lag features and derived metrics for better predictions
3. **Model Comparison**: Evaluated multiple algorithms (Linear Regression, Random Forest, Gradient Boosting)
4. **Optimization**: Performed hyperparameter tuning for the best performing model
5. **Deployment Ready**: Created prediction function and saved model for production use

The model can be used to:
- Predict energy consumption based on system metrics
- Optimize resource allocation to minimize energy usage
- Plan capacity and energy budgets
- Identify anomalous energy consumption patterns